In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel(r"./上證指數.xlsx")
df

,名稱,代碼,日期,開盤,最高,最低,收盤,漲跌
0,上證指數,980001,2017-01-03,3105.31,3136.46,3105.31,3135.92,1
1,上證指數,980001,2017-01-04,3133.79,3160.10,3130.11,3158.79,1
2,上證指數,980001,2017-01-05,3157.91,3168.50,3154.28,3165.41,1
3,上證指數,980001,2017-01-06,3163.78,3172.03,3153.03,3154.32,0
4,上證指數,980001,2017-01-09,3148.53,3173.14,3147.74,3171.24,1
...,...,...,...,...,...,...,...,...
177,上證指數,980001,2017-09-21,3364.70,3377.88,3356.88,3357.81,0
178,上證指數,980001,2017-09-22,3347.16,3356.45,3334.98,3352.53,0
179,上證指數,980001,2017-09-25,3344.59,3350.96,3334.94,3341.55,0
180,上證指數,980001,2017-09-26,3336.35,3347.16,3332.60,3343.58,1


In [ ]:
import torch
import torch.utils.data as Data
import numpy as np

In [ ]:
class MyDataset(Data.Dataset):
    def __init__(self, file_path, seq_length, train=True, transform=None):
        df = pd.read_excel(r"./上證指數.xlsx")
        df_len = len(df) - 1
        df_len -= df_len % seq_length
        total_size = df_len // seq_length
        train_size = int(total_size * 0.8)

        f = df.iloc[: (train_size * seq_length), 3:7]
        l = df.iloc[1 : (train_size * seq_length) + 1, 7:]
        data_size = train_size

        if train == False:
            f = df.iloc[(train_size * seq_length) : df_len, 3:7]
            l = df.iloc[(train_size * seq_length) + 1 : df_len + 1, 7:]
            data_size = total_size - train_size

        self.features = np.array(f).astype("float32").reshape(data_size, seq_length, 4)
        self.labels = np.array(l).astype("int").reshape(data_size, seq_length, 1)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        X = self.features[idx]
        Y = self.labels[idx]

        if self.transform:
            X = self.transform(X)
            Y = self.transform(Y)

        return X, Y

In [ ]:
def toTensor(x):
    return torch.tensor(x)

In [ ]:
train_dataset = MyDataset(
    file_path=r"./上證指數.xlsx", seq_length=5, train=True, transform=toTensor
)

test_dataset = MyDataset(
    file_path=r"./上證指數.xlsx", seq_length=5, train=False, transform=toTensor
)

In [ ]:
features, labels = train_dataset[2:9]
print(features.shape)
print(labels.shape)

torch.Size([7, 5, 4])
torch.Size([7, 5, 1])


In [ ]:
train_loader = Data.DataLoader(dataset=train_dataset, batch_size=5, shuffle=True)
test_loader = Data.DataLoader(dataset=test_dataset, batch_size=5, shuffle=False)

In [ ]:
for i, (seqs, labels) in enumerate(train_loader):
    print(seqs.shape)
seqs

torch.Size([5, 5, 4])
torch.Size([5, 5, 4])
torch.Size([5, 5, 4])
torch.Size([5, 5, 4])
torch.Size([5, 5, 4])
torch.Size([3, 5, 4])


tensor([[[3233.0901, 3242.6599, 3226.8201, 3242.4099],
         [3240.5300, 3245.3000, 3230.6101, 3240.6599],
         [3233.7000, 3233.8701, 3205.2800, 3216.7500],
         [3213.7300, 3222.3201, 3208.4500, 3212.7600],
         [3209.4500, 3237.1201, 3193.1599, 3237.0200]],

        [[3134.5901, 3149.5300, 3131.2200, 3142.5500],
         [3137.6499, 3151.4700, 3133.1899, 3149.5500],
         [3149.2200, 3163.1001, 3148.9099, 3159.1699],
         [3160.0801, 3162.6799, 3136.0100, 3140.1699],
         [3143.0901, 3158.8401, 3135.3899, 3156.9800]],

        [[3154.3999, 3159.5400, 3140.0400, 3153.0901],
         [3148.0901, 3167.4500, 3132.0300, 3166.9800],
         [3164.6899, 3186.8401, 3162.5701, 3183.1799],
         [3183.0100, 3205.0500, 3182.8000, 3196.7000],
         [3198.9900, 3219.4099, 3198.9900, 3216.8401]]])

In [ ]:
input_size = 4
hidden_size = 32
num_layers = 1
num_classes = 1

In [ ]:
import torch.nn as nn
from torch.autograd import Variable

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.normal = nn.BatchNorm1d(5, affine=True)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size))
        c0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size))
        out = self.normal(x)
        out, _ = self.lstm(out, (h0, c0))
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

In [ ]:
rnn = RNN(input_size, hidden_size, num_layers, num_classes)
rnn

RNN(
  (normal): BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (lstm): LSTM(4, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [ ]:
output = rnn(seqs)
output

tensor([[[0.5137],
         [0.5158],
         [0.5169],
         [0.5158],
         [0.5149]],

        [[0.5129],
         [0.5181],
         [0.5214],
         [0.5255],
         [0.5241]],

        [[0.5131],
         [0.5173],
         [0.5208],
         [0.5248],
         [0.5250]]], grad_fn=<SigmoidBackward0>)

In [ ]:
import torch.optim as optim

In [ ]:
loss_func = nn.BCELoss()
optimizer = optim.Adam(rnn.parameters(), lr=0.001)

In [ ]:
for epoch in range(800):
    for i, (seqs, labels) in enumerate(train_loader):
        seqs = Variable(seqs)
        labels = Variable(labels)
        optimizer.zero_grad()
        output = rnn(seqs)
        loss = loss_func(output, labels.float())
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print("Epoch: %d, batch: %d, loss: %f" % (epoch + 1, i + 1, loss.data))

Epoch: 100, batch: 1, loss: 0.662186
Epoch: 100, batch: 2, loss: 0.560024
Epoch: 100, batch: 3, loss: 0.663438
Epoch: 100, batch: 4, loss: 0.698814
Epoch: 100, batch: 5, loss: 0.624139
Epoch: 100, batch: 6, loss: 0.525051
Epoch: 200, batch: 1, loss: 0.677817
Epoch: 200, batch: 2, loss: 0.667798
Epoch: 200, batch: 3, loss: 0.540516
Epoch: 200, batch: 4, loss: 0.573737
Epoch: 200, batch: 5, loss: 0.654268
Epoch: 200, batch: 6, loss: 0.620987
Epoch: 300, batch: 1, loss: 0.621231
Epoch: 300, batch: 2, loss: 0.594067
Epoch: 300, batch: 3, loss: 0.537549
Epoch: 300, batch: 4, loss: 0.644535
Epoch: 300, batch: 5, loss: 0.653901
Epoch: 300, batch: 6, loss: 0.623535
Epoch: 400, batch: 1, loss: 0.611644
Epoch: 400, batch: 2, loss: 0.617713
Epoch: 400, batch: 3, loss: 0.529143
Epoch: 400, batch: 4, loss: 0.626486
Epoch: 400, batch: 5, loss: 0.720745
Epoch: 400, batch: 6, loss: 0.685689
Epoch: 500, batch: 1, loss: 0.614187
Epoch: 500, batch: 2, loss: 0.563572
Epoch: 500, batch: 3, loss: 0.596292
E

In [ ]:
correct = 0
total = 0

for seqs, labels in test_loader:
    seqs = Variable(seqs)
    output = rnn(seqs)
    predicted = output.gt(0.5)
    total += labels.size(0) * labels.size(1)
    correct += (predicted == labels).sum()

print("Accuracy: %.3f %%" % (100 * float(correct) // float(total)))

Accuracy: 42.000 %
